In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import gc

In [ ]:
NULL_DIRECTORY = "../output/NULL"
NONE_DIRECTORY = "../output/NONE"
DISEASE_DIRECTORY = f"../output/BIPOLAR"

In [ ]:
# DGIDB index -> MSIGDB index for the null run, None when the gene is DGIDB-only.
# NULL is a copy of BIPOLAR, so this is the real mapping, not a randomized one.
with open(f"{NULL_DIRECTORY}/dgidb_to_msigdb_indices_dict.json") as file:
    dgidb_to_msigdb_dict = json.load(file)

# The DGIDB genes shared with the MSIGDB layer, in DGIDB index order. The rows of
# P_t_avg_dgidb_rows.npy are in that same order, so the mask and this list describe the
# same genes in the same order.
shared_mask = np.array(
    [midx is not None for midx in dgidb_to_msigdb_dict.values()], dtype=bool
)
shared_msigdb_rows = [
    midx for midx in dgidb_to_msigdb_dict.values() if midx is not None
]

# Genes that only exist in DGIDB. They take the leading rows of the aggregated index
# space, so this is also the shift between an MSIGDB index and its aggregated column.
num_dgidb_only = int((~shared_mask).sum())

# The same genes in the aggregated index space, where the DGIDB-only genes come first
# and every MSIGDB gene is pushed back by that count. A shared gene has a single
# aggregated index, so this indexes both the rows and the columns of P_t_avg.npy.
shared_agg_rows = [midx + num_dgidb_only for midx in shared_msigdb_rows]

len(shared_msigdb_rows), len(shared_agg_rows), num_dgidb_only

In [ ]:
# load ddm
profiles_msigdb = np.load(f"{NONE_DIRECTORY}/P_t_avg.npy",mmap_mode="r")
profiles_real = np.load(f"{DISEASE_DIRECTORY}/P_t_avg.npy",mmap_mode="r")

num_msigdb_genes = profiles_msigdb.shape[0]

In [ ]:
dgidb_profiles_msigdb = profiles_msigdb[shared_msigdb_rows, :]
dgidb_profiles_real = profiles_real[shared_agg_rows, num_dgidb_only:]

# Compute statistics

In [ ]:
def relative_difference(profiles, reference):
    numerator = np.linalg.norm(profiles - reference, axis=1)
    denominator = np.linalg.norm(reference, axis=1) + 1e-12
    return numerator / denominator

In [ ]:
null_paths = sorted(Path(NULL_DIRECTORY).glob("P_t_avg_dgidb_shared_rows_*.npy"))

rel_diff_null_runs = np.array([
    relative_difference(np.load(path, mmap_mode="r")[:, num_dgidb_only:], dgidb_profiles_msigdb)
    for path in null_paths
])

rel_diff_null_runs.shape

In [ ]:
S_real = relative_difference(dgidb_profiles_real, dgidb_profiles_msigdb)

In [ ]:
S_null_runs = np.array([])
for path in null_paths:
    dgidb_profiles_null = np.load(path, mmap_mode="r")[:, num_dgidb_only:]
    S_null = relative_difference(dgidb_profiles_null, dgidb_profiles_msigdb)
    S_null_runs = np.append(S_null_runs, S_null)

S_null_runs.shape